In [1]:
# %% [1] 환경 설정 및 필수 라이브러리 임포트
import os
import sys
import pandas as pd
import numpy as np
import json
import requests
import time
from datetime import datetime, timedelta
# Use %pip in notebooks so the package is installed into the running environment
from dotenv import load_dotenv
from elasticsearch import Elasticsearch


# 프로젝트 루트 경로 설정:
# ai_pipeline 패키지를 포함한 디렉터리를 자동으로 찾아 sys.path에 추가합니다.
def _find_project_root(target_pkg='ai_pipeline'):
    p = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(p, target_pkg)):
            # p 폴더 내부에 target_pkg가 있으면 p를 프로젝트 루트로 사용
            return p
        newp = os.path.dirname(p)
        if newp == p:
            # 최상위까지 찾아도 없으면 현재 작업 디렉터리를 그대로 사용
            return os.path.abspath(os.getcwd())
        p = newp

PROJECT_ROOT = _find_project_root('ai_pipeline')
sys.path.append(PROJECT_ROOT)

# .env 파일 로드
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# 기존 모듈 활용 (Feature Engineering 등)
from ai_pipeline.boosting_model.realtime_feature_loader import RealtimeFeatureLoader
from ai_pipeline.boosting_model.train import StackingEnsemble

# 설정 값
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
ES_HOST = "http://localhost:9200"

# 학습에 사용할 날짜 리스트 (파일이 실제로 존재하는지 확인 필요)
TARGET_DATES = [
    "20251120", "20251121", "20251124", "20251125",
    "20251126", "20251127", "20251202", "20251203",
    "20251204"
]

print(f"🚀 작업 시작: {len(TARGET_DATES)}일치 데이터를 학습합니다.")

# %% [2] KIS 모의투자 API 클래스 (주문용)
class KISMockTrader:
    def __init__(self):
        self.app_key = os.getenv("KIS_MOCK_APP_KEY")
        self.app_secret = os.getenv("KIS_MOCK_APP_SECRET")
        self.acc_no = os.getenv("KIS_MOCK_ACCOUNT_NO") # 계좌번호 앞 8자리
        self.base_url = "https://openapivts.koreainvestment.com:29443"
        self.access_token = None

        if not all([self.app_key, self.app_secret, self.acc_no]):
            print("❌ [KIS] .env에 모의투자 계좌 정보가 없습니다.")
        else:
            self.auth()

    def auth(self):
        """접근 토큰 발급"""
        headers = {"content-type": "application/json"}
        body = {
            "grant_type": "client_credentials",
            "appkey": self.app_key,
            "appsecret": self.app_secret
        }
        res = requests.post(f"{self.base_url}/oauth2/tokenP", headers=headers, data=json.dumps(body))
        if res.status_code == 200:
            self.access_token = res.json()["access_token"]
            print(f"✅ [KIS] 토큰 발급 완료 (유효기간: {res.json()['access_token_token_expired']})")
        else:
            print(f"❌ [KIS] 토큰 발급 실패: {res.text}")

    def get_common_headers(self, tr_id):
        return {
            "content-type": "application/json",
            "authorization": f"Bearer {self.access_token}",
            "appkey": self.app_key,
            "appsecret": self.app_secret,
            "tr_id": tr_id
        }

    def buy_limit(self, stock_code, price, qty):
        """지정가 매수"""
        if not self.access_token: return
        headers = self.get_common_headers("VTTC0802U") # 모의투자 매수 TR ID
        body = {
            "CANO": self.acc_no,
            "ACNT_PRDT_CD": "01",
            "PDNO": stock_code,
            "ORD_DVSN": "00", # 00: 지정가, 01: 시장가
            "ORD_QTY": str(qty),
            "ORD_UNPR": str(price)
        }
        res = requests.post(f"{self.base_url}/uapi/domestic-stock/v1/trading/order-cash", headers=headers, data=json.dumps(body))
        print(f" 🛒 [매수주문] {stock_code} {qty}주 @ {price}원 -> 결과: {res.json().get('msg1')}")
        return res.json()

    def sell_market(self, stock_code, qty):
        """시장가 매도"""
        if not self.access_token: return
        headers = self.get_common_headers("VTTC0801U") # 모의투자 매도 TR ID
        body = {
            "CANO": self.acc_no,
            "ACNT_PRDT_CD": "01",
            "PDNO": stock_code,
            "ORD_DVSN": "01", # 시장가
            "ORD_QTY": str(qty),
            "ORD_UNPR": "0"
        }
        res = requests.post(f"{self.base_url}/uapi/domestic-stock/v1/trading/order-cash", headers=headers, data=json.dumps(body))
        print(f" 👋 [매도주문] {stock_code} {qty}주 (시장가) -> 결과: {res.json().get('msg1')}")
        return res.json()

# %% [3] 데이터 로더 함수 (뉴스 + CSV 병합) - 가격 정보 보존 버전
def load_data_and_merge_news(date_str, data_dir, es_client):
    """특정 날짜의 CSV를 읽고, 해당 날짜의 뉴스 감성점수를 매핑"""
    csv_path = os.path.join(data_dir, f"{date_str}.csv")
    if not os.path.exists(csv_path):
        print(f"   ⚠️ 파일 없음: {csv_path}")
        return None

    # 1. 시세 데이터 로드 (메서드 분리 호출로 데이터 보존)
    loader = RealtimeFeatureLoader(csv_path)

    # (1) 전체 데이터 로드
    df = loader.load_and_preprocess()
    if df.empty: return None

    # (2) 기술적 지표 생성 (여기서 stck_prpr이 유지됨)
    df = loader.create_technical_features(df)
    if df.empty: return None

    # (3) 결측치 제거
    df = df.dropna()
    if len(df) == 0: return None

    # (4) 필요한 컬럼만 선택하되, 'stck_prpr'은 꼭 챙김!
    feature_cols = [
        'prdy_ctrt', 'price_change_1', 'price_change_5', 'price_change_10',
        'price_vs_ma5', 'price_vs_ma20', 'tr_amount_change', 'spread', 'spread_pct',
        'buy_pressure', 'buy_strength', 'volatility_5', 'volatility_10',
        'momentum_5', 'momentum_10'
    ]

    # 없는 피처는 0으로 채움 (안전장치)
    for c in feature_cols:
        if c not in df.columns: df[c] = 0.0

    # [핵심] 모델 피처 + 타겟 + 종목코드 + **현재가(stck_prpr)**
    keep_cols = feature_cols + ['target', 'stock_code', 'stck_prpr']

    # stck_prpr이 혹시 없을 경우를 대비
    if 'stck_prpr' not in df.columns:
        df['stck_prpr'] = 0

    df_result = df[keep_cols].copy()

    # 2. ES에서 해당 날짜 뉴스 감성 가져오기
    dt_obj = datetime.strptime(date_str, "%Y%m%d")
    start_dt = dt_obj.replace(hour=0, minute=0, second=0).isoformat()
    end_dt = dt_obj.replace(hour=23, minute=59, second=59).isoformat()

    query = {
        "range": {
            "timestamp": {
                "gte": start_dt,
                "lte": end_dt
            }
        }
    }

    # 종목별 감성 평균 집계
    try:
        resp = es_client.search(
            index="news_articles",
            body={
                "size": 0,
                "query": query,
                "aggs": {
                    "by_stock": {
                        "terms": {"field": "related_stocks.keyword", "size": 1000},
                        "aggs": {"avg_sentiment": {"avg": {"field": "sentiment_score"}}}
                    }
                }
            }
        )
        sentiment_map = {}
        if 'aggregations' in resp:
            for bucket in resp['aggregations']['by_stock']['buckets']:
                code = bucket['key']
                score = bucket['avg_sentiment']['value']
                sentiment_map[code] = score if score else 0.0
    except Exception as e:
        print(f"   [ES Error] 뉴스 데이터 로드 실패 (0점으로 처리): {e}")
        sentiment_map = {}

    # 3. DataFrame에 감성 점수 매핑
    df_result['sentiment_score'] = df_result['stock_code'].map(sentiment_map).fillna(0.0)

    return df_result

# %% [4] 통합 학습 데이터셋 생성
es = Elasticsearch(ES_HOST)
all_data_list = []

print("\n📊 데이터 로딩 및 전처리 시작...")
for date_str in TARGET_DATES:
    print(f"   Processing {date_str}...", end=" ")
    df_day = load_data_and_merge_news(date_str, DATA_DIR, es)
    if df_day is not None:
        print(f"✅ Rows: {len(df_day)}")
        all_data_list.append(df_day)
    else:
        print("❌ Skipped")

if not all_data_list:
    raise ValueError("학습할 데이터가 없습니다!")

# 전체 데이터 병합
full_df = pd.concat(all_data_list, ignore_index=True)
print(f"\n📚 전체 데이터셋 크기: {full_df.shape}")
print(f"   상승(1): {full_df['target'].sum()} / 하락(0): {len(full_df) - full_df['target'].sum()}")

# %% [5] 모델 학습 (XGBoost)
print("\n🧠 모델 학습 시작...")

# 학습에 불필요한 컬럼 제거
train_cols = [c for c in full_df.columns if c not in ['target', 'stock_code', 'date', 'timestamp', 'stck_prpr']]
print(f"   학습 피처({len(train_cols)}개): {train_cols}")

X = full_df[train_cols]
y = full_df['target']

# 모델 초기화 및 학습
model = StackingEnsemble()
model.feature_names = train_cols # 피처 이름 저장 (중요)
model.train(X, y) # 내부에서 Train/Val split 수행

print("🎉 모델 학습 완료!")

# %% [6] 실전(모의) 매매 시뮬레이션
# 시나리오: 가장 마지막 날짜(12월 5일)의 '마지막 데이터'를 현재 시점이라고 가정하고 매매 판단
print("\n🤖 AI 트레이더 가동 (Simulation Mode)")

last_date_df = all_data_list[-1] # 12월 5일 데이터
# 각 종목별 가장 마지막 row만 추출 (최신 상태)
current_market_status = last_date_df.drop_duplicates(subset=['stock_code'], keep='last').copy()

print(f"   현재 포착된 종목 수: {len(current_market_status)}개")

# 예측 수행
X_live = current_market_status[train_cols]
# 에러 방지를 위해 모델이 아는 피처만 필터링
X_live = X_live[model.feature_names]

# 확률 예측
probs = model.predict_proba(X_live)[:, 1] # 상승 확률
current_market_status['ai_score'] = probs * 100

# 매수 대상 선정 (AI Score 85점 이상)
buy_candidates = current_market_status[current_market_status['ai_score'] >= 0].sort_values('ai_score', ascending=False)

print("\n📋 [AI 추천 종목 Top 5]")
print(buy_candidates[['stock_code', 'ai_score', 'stck_prpr']].head(5))

# %% [7] 주문 실행 (KIS Mock API)
# 주의: 장 운영 시간이 아니면 실패할 수 있습니다.
kis = KISMockTrader()

if not buy_candidates.empty:
    print("\n💰 자동 주문 실행 중...")

    # 예산 설정 (예: 종목당 10주씩 매수)
    for idx, row in buy_candidates.head(3).iterrows(): # 상위 3개만
        code = row['stock_code']
        score = row['ai_score']
        price = row['stck_prpr'] # 현재가

        # 목표가: 현재가보다 1호가 위로 지정 (즉시 체결 유도)
        # 틱 단위 계산 로직은 생략하고 단순하게 +50원 처리
        target_price = int(price)

        print(f"   👉 발견! [{code}] 점수: {score:.1f}점 -> 매수 시도")

        # 실제 API 호출
        kis.buy_limit(stock_code=code, price=target_price, qty=10)
        time.sleep(0.5) # API 제한 고려
else:
    print("\n😴 살만한 종목이 없습니다. (기준 점수 미달)")

print("\n🏁 모든 프로세스 종료.")

🚀 작업 시작: 9일치 데이터를 학습합니다.

📊 데이터 로딩 및 전처리 시작...
   Processing 20251120...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251120.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 14463
   Processing 20251121...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251121.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 14680
   Processing 20251124...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251124.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 14402
   Processing 20251125...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251125.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 14938
   Processing 20251126...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251126.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 11446
   Processing 20251127...  [Loader] 초기화됨 (New Version Check): c:\Users\user\project\MyEggBasket-AI\data\20251127.csv

 체결 정보 CSV 로딩 중...
✅ Rows: 11446
   Processing 20251202...  [Loader] 초기화됨 (New V